# 16. Real Paper Reproduction Case Study — Noise2Void for Microscopy Denoising

**Paper:** Krull, Buchholz & Jug, *Noise2Void — Learning Denoising from Single Noisy Images*, CVPR (2019)  
**Authors' code:** `https://github.com/juglab/n2v`  
**Goal:** reproduce a legacy self-supervised denoising method, understand the statistical assumptions, then decide whether to preserve the original code, use maintained tooling, or reimplement transparently.

CARE and N2V answer different supervision situations:

```text
paired clean/high-quality target available? YES -> CARE-like supervised route
                                     NO
                                     ↓
noise compatible with blind-spot assumptions? YES -> N2V-like route
                                         NO -> choose another strategy
```

## Mind map

```mermaid
mindmap
  root((Noise2Void paper to code))
    Assumptions
      Signal predictable from context
      Noise conditionally independent
    Authors code
      juglab n2v
      TensorFlow
      CSBDeep
      Legacy environment
    Mechanism
      Select pixels
      Hide center value
      Predict from neighbors
      Loss only on hidden pixels
    Reproduction
      Pin versions
      Run official example
      Record config
      Compare output
    Optical risks
      Shot noise
      Fixed pattern
      Stripes
      Speckle
      Reconstruction artifacts
```


## Step 1 — Decide whether N2V is scientifically plausible

N2V does **not** work just because you lack clean targets. Its core assumptions matter.

### Assumption A — signal is spatially predictable

Neighboring pixels should contain information about the true center-pixel signal.

### Assumption B — noise is conditionally independent across pixels

Neighbors should *not* reveal the exact random noise realization at the center.

Favorable examples:
- shot-noise-dominated fluorescence;
- approximately independent camera read noise.

Risky examples:
- row/column banding;
- fixed-pattern noise;
- interpolation artifacts;
- correlated reconstruction artifacts;
- coherent/signal-dependent speckle.

For OCT, do **not** assume N2V validity from appearance alone; speckle is tied to coherent interference and tissue microstructure.

## Step 2 — Audit the official repository today

The official `juglab/n2v` repository now warns that the old package is being phased out, that modern development has moved to newer tooling such as CAREamics, and that the legacy code is not compatible with TensorFlow 2.16. Its README provides a Python 3.9-era compatibility route.

That means you need two goals:

**Reference reproduction:** preserve the old TensorFlow environment long enough to reproduce behavior.  
**New research pipeline:** after reproduction, use a maintained implementation or a tested PyTorch reimplementation if appropriate.


## Step 3 — Legacy reproduction ladder

```mermaid
flowchart TD
  A[Freeze official repo commit] --> B[Create compatible legacy env]
  B --> C[Run smallest official example]
  C --> D[Check axes + patch shape]
  D --> E[Reproduce inference]
  E --> F[Reproduce training]
  F --> G[Record N2V config]
  G --> H[Compare reference behavior]
  H --> I[Only then port/adapt]
```

Shell strategy:

```bash
git clone https://github.com/juglab/n2v.git
cd n2v
git rev-parse HEAD

conda create -n n2v-repro python=3.9
conda activate n2v-repro

# Follow the repository's tested TensorFlow guidance.
python -c "import tensorflow as tf; print(tf.__version__)"
```

Start with an official example notebook before your own data. The repository includes 2-D and 3-D examples. After the software path works, move to a public fluorescence dataset or your own optical data.

### Source-map questions

| Concept | What to locate |
|---|---|
| U-Net construction | network builder |
| blind-spot pixel percentage | N2V config |
| pixel replacement/manipulator | N2V utilities/data wrapper |
| masked target channel | training data preparation |
| loss | masked N2V loss |
| patch axes | data generation |
| prediction axes | model predict call |

Key question: **where does the implementation stop the model from learning an identity mapping at supervised pixels?**


## Step 4 — Self-contained teaching reimplementation of the core idea

This PyTorch example is intentionally simplified. It demonstrates the masking logic, not the full authors' package.

Training sees only one noisy image:

```text
noisy image
   ↓
randomly choose pixels
   ↓
replace selected center values using neighboring context
   ↓
predict original noisy values
   ↓
compute loss ONLY at masked locations
```

For teaching, we keep a latent clean image only to evaluate whether denoising helped. The clean image is **not used as a training target**.


In [ ]:
import math, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(11)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

def scene(size=48, seed=0):
    g=torch.Generator().manual_seed(seed)
    y,x=torch.meshgrid(torch.linspace(-1,1,size),torch.linspace(-1,1,size),indexing="ij")
    img=torch.zeros(size,size)
    for _ in range(12):
        cx=torch.rand(1,generator=g).item()*1.6-0.8
        cy=torch.rand(1,generator=g).item()*1.6-0.8
        s=0.03+0.08*torch.rand(1,generator=g).item()
        a=0.25+0.75*torch.rand(1,generator=g).item()
        img += a*torch.exp(-((x-cx)**2+(y-cy)**2)/(2*s*s))
    return img/img.max().clamp_min(1e-8)

clean=scene(seed=4)
noisy=(torch.poisson(clean*20)/20 + 0.025*torch.randn_like(clean)).clamp(0,1)
img=noisy[None,None].to(device)
print(img.shape)


In [ ]:
def mask_batch(x, frac=0.08):
    target=x.clone()
    mask=torch.rand_like(x)<frac
    # simplified neighbor replacement
    replacement=torch.roll(x,shifts=1,dims=-1)
    xm=x.clone()
    xm[mask]=replacement[mask]
    return xm,target,mask

class ContextNet(nn.Module):
    def __init__(self,w=24):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(1,w,3,padding=1),nn.ReLU(),
            nn.Conv2d(w,w,3,padding=1),nn.ReLU(),
            nn.Conv2d(w,w,3,padding=1),nn.ReLU(),
            nn.Conv2d(w,1,3,padding=1)
        )
    def forward(self,x): return self.net(x)

model=ContextNet().to(device)
opt=torch.optim.Adam(model.parameters(),lr=2e-3)

for step in range(100):
    xm,t,m=mask_batch(img)
    pred=model(xm)
    loss=F.mse_loss(pred[m],t[m])
    opt.zero_grad(); loss.backward(); opt.step()

with torch.no_grad():
    den=model(img).cpu()[0,0].clamp(0,1)

def psnr(a,b):
    mse=torch.mean((a-b)**2).item()
    return 10*math.log10(1/max(mse,1e-12))

print("noisy PSNR:",round(psnr(noisy,clean),2))
print("teaching N2V-like PSNR:",round(psnr(den,clean),2))


## Step 5 — Understand what the toy example does / does not establish

**It demonstrates:** random blind-spot masking, neighbor replacement, masked-pixel loss, and training without a clean target.

**It does not establish:** the exact N2V architecture/manipulator/config, the paper's numerical result, or suitability for your detector.

For faithful reproduction, replace components in order:

```text
toy masking -> authors N2V data wrapper/manipulator
toy network -> authors U-Net/config
toy image   -> official example data
toy metric  -> authors evaluation
```

## Step 6 — Test a failure mode: correlated noise

Suppose a camera adds row-wise stripes. Neighboring pixels now reveal the artifact itself, so a context model may treat the stripe as signal.

Likewise, coherent OCT speckle is not equivalent to independent Gaussian noise. The right question is not "does N2V make it smoother?" but:

- are weak structures preserved?
- are quantitative intensities preserved?
- does downstream segmentation/tracking improve?
- are artifacts reduced without changing biology?
- does the method generalize across acquisitions/specimens?

## Optical decision table

| Data situation | Initial N2V judgement |
|---|---|
| low-photon fluorescence + mostly independent noise | reasonable candidate |
| LSFM shot noise | plausible, validate structures |
| fixed-pattern camera noise | assumption risk |
| scan-line stripes | correct/model structure first |
| MUSE camera noise | possible after separating system artifacts |
| OCT speckle | do not assume validity; physics differs |

## Step 7 — Reproduce first, modernize second

For an aging codebase:

1. freeze old repository + environment;
2. reproduce an official example;
3. record config, axes, patch shape, masked-pixel percentage and manipulator;
4. understand the source code;
5. then port to PyTorch or use maintained modern tooling;
6. compare old and new implementations on the same images.

That is a **cross-framework reproduction**, not a casual rewrite.

### Reproduction diary

```text
paper:
repo commit:
Python/TensorFlow:
official example:
data:
axes:
patch size:
n2v_perc_pix:
manipulator:
network config:
steps/epochs:
validation:
prediction axes:
reference output:
my output:
errors/warnings:
scientific assumptions:
decision: REUSE / ADAPT / REIMPLEMENT / REJECT
```


## Final strategy learned from CARE + N2V

These two papers give you a reusable supervision decision tree:

```mermaid
flowchart TD
  A[Optical imaging problem] --> B{Trustworthy paired target?}
  B -->|Yes| C[CARE-like supervised restoration]
  B -->|No| D{Noise compatible with self-supervised assumptions?}
  D -->|Yes| E[N2V-like blind-spot strategy]
  D -->|No| F[Structured-noise / physics-aware / alternative method]
  C --> G[Reproduce authors code first]
  E --> G
  F --> H[Build problem-specific baseline]
  G --> I[Validate on independent specimens]
  I --> J[Only then optimize or modernize]
```

The architecture is not the first decision. **The measurement process, supervision availability, and noise physics are.**
